## 1. Carga y configuración inicial

Antes de comenzar, vamos a importar todas las librerías a utilizar con el objetivo de tener la libreta organizada desde el principio:

In [ ]:
# Standard
import warnings

# Third party
import matplotlib.pyplot as plt
import missingno as msno
import pandas as pd
import seaborn as sns
from sklearn import set_config
from sklearn.model_selection import train_test_split

import json

Además, configuramos la visualización de las gráficas:

In [ ]:
%matplotlib inline
figsize = (12, 7)  # Width, height in inches
palette = "colorblind"  # Palette definition
plt.rcParams["figure.figsize"] = figsize
sns.set_palette(palette)

A su vez, fijamos una semilla para que los experimentos sean reproducibles:

In [ ]:
random_state = seed = 41686

Vamos a indicarle a `scikit-learn` que la salida de sus transformadores sea un `DataFrame` de `pandas` siempre que sea posible, en lugar de un `array` de `NumPy` (que es el comportamiento por defecto). Hacemos esto ahora porque, aunque no afecta al modelo final, hace que el **proceso de definir y depurar los pasos de preprocesamiento sea mucho más cómodo**. Nos permitirá inspeccionar la salida de nuestros `Pipeline` en cualquier etapa y ver un `DataFrame` con nombres de columna legibles, facilitando la verificación de que nuestra lógica es correcta, en lugar de tener que descifrar un `array` numérico anónimo.

In [ ]:
transform_output = "pandas"  # Pandas output
set_config(transform_output=transform_output)

Por último, filtramos los mensajes de advertencia para evitar salidas demasiado largas en la libreta:

In [ ]:
action = "ignore"  # Never print matching warnings
warnings.filterwarnings(action)

## 2. Carga de datos

Vamos a cargar el conjunto de datos [`Bank Marketing`](https://archive.ics.uci.edu/dataset/222/bank+marketing). Este conjunto registra los resultados de una campaña completa de marketing directo (llamadas telefónicas) de un banco portugués, diseñada para que los clientes suscribieran un depósito a plazo fijo.


El problema de negocio es **optimizar las campañas de marketing** identificando de antemano qué clientes tienen mayor probabilidad de suscribirse al producto. Desde la perspectiva del aprendizaje automático, este es un problema clásico de **clasificación binaria supervisada**. El objetivo final es construir un modelo (en nuestro caso, una **regresión logística**) que aprenda de las características del cliente y de la campaña para predecir si un cliente dirá "sí" o "no".

La variable que intentamos predecir es `y`, categórica, con valores `"yes"` (el cliente se suscribió) y `"no"` (el cliente no se suscribió).

Las variables que usaremos como predictoras se pueden agrupar lógicamente:

#### Datos del cliente

* `age` (numérica): Edad del cliente.
* `job` (categórica): Tipo de trabajo.
* `marital` (categórica): Estado civil.
* `education` (categórica): Nivel educativo.
* `default` (categórica): ¿Tiene crédito en impago?
* `balance` (numérica): Saldo medio anual, en euros.
* `housing` (categórica): ¿Tiene un préstamo hipotecario?
* `loan` (categórica): ¿Tiene un préstamo personal?

#### Datos del último contacto

* `contact` (categórica): Tipo de comunicación del contacto.
* `day` (numérica): Día del último contacto.
* `month` (categórica): Mes del último contacto.
* `duration` (numérica): Duración del último contacto, en segundos.

#### Historial de campañas

* `campaign` (numérica): Número de contactos realizados durante esta campaña para este cliente.
* `pdays` (numérica): Días transcurridos desde el último contacto de una campaña previa.
* `previous` (numérica): Número de contactos realizados antes de esta campaña para este cliente.
* `poutcome` (categórica): Resultado de la campaña previa.

In [ ]:
file = "/home/victor/Descargas/xmas_merged_data.json"
parameters = []  # Example: ["cbmac_details.cbmac_load", "buffer_usage.average"]

print("Processing Data...")

batch_size = 100_000  # número de líneas por lote
batches = []  # para acumulación temporal

for i, line in enumerate(open(file, "r", encoding="utf-8"), start=1):
    line = line.strip()
    if not line:
        continue
    try:
        obj = json.loads(line)
        batches.append(obj)
    except json.JSONDecodeError:
        continue

    # cada cierto número de líneas, convertimos y guardamos en disco o acumulamos
    if len(batches) >= batch_size:
        batch_df = pd.json_normalize(batches)
        if i == batch_size:
            data = batch_df  # primer lote
        else:
            data = pd.concat([data, batch_df], ignore_index=True)
        batches.clear()
        print(f"Procesadas {i:,} líneas...")

# procesar el resto
if batches:
    batch_df = pd.json_normalize(batches)
    data = pd.concat([data, batch_df], ignore_index=True) if "data" in locals() else batch_df

try:
    for col in data.select_dtypes(include=["int64", "float64"]).columns:
        if data[col].min() >= -32768 and data[col].max() <= 32767:
            data[col] = data[col].astype("Int16")
        else:
            print(f"⚠️  Columna {col} no cabe en int16")
except Exception as e:
    print(f"⚠️  Error al convertir tipos numéricos: {e}")

In [ ]:
print(data.head())
print(data.columns)

### 2.1. División estratégica: conjuntos de entrenamiento y prueba

Antes de transformar o analizar en profundidad cualquier dato, debemos realizar el paso metodológico más importante: **separar nuestro conjunto de datos en entrenamiento y prueba**.

#### **¿Por qué ahora?**

Para garantizar una evaluación honesta y robusta de nuestro modelo.

* El conjunto de **entrenamiento** (`X_train`, `y_train`) será utilizado para *todo*: análisis exploratorio, imputación de valores faltantes, detección de valores extremos, escalado y, por supuesto, el entrenamiento del modelo.
* El conjunto de **prueba** (`X_test`, `y_test`) se mantendrá "guardado bajo llave" y completamente intacto. No lo miraremos ni lo usaremos para tomar ninguna decisión de preprocesamiento.

De esta forma, simulamos un escenario real donde el modelo se enfrenta a datos que nunca ha visto, previniendo el ***data leakage*** (fuga de datos). Si calculáramos, por ejemplo, la media para escalar usando el conjunto de datos completo, el conjunto de prueba ya habría "contaminado" nuestro preprocesamiento.

Usaremos una división estratificada (`stratify = y`) para asegurar que la proporción de `"yes"` y `"no"` sea la misma tanto en el conjunto de entrenamiento como en el de prueba, lo cual es vital en problemas de clasificación (especialmente si están desbalanceados).

In [ ]:
test_size = 0.2  # Proportion of the dataset to include in the test split

X_train, X_test = train_test_split(data, test_size=test_size, random_state=random_state)

In [ ]:
n = 5
X_train.sample(n, random_state=random_state)

In [ ]:
X_test.sample(n, random_state=random_state)

Las proporciones de la clase objetivo son casi idénticas en ambos conjuntos.

**A partir de este punto, guardamos `X_test` y `y_test` y nos olvidamos de ellos hasta la evaluación final.**

Todo el análisis y preprocesamiento se realizará **únicamente** sobre `X_train` y `y_train`.

## 3. Análisis exploratorio de datos

Ahora que hemos aislado nuestros datos de prueba, comienza el verdadero trabajo de preprocesamiento. Todo el análisis, decisiones y transformaciones se basarán **únicamente** en `X_train` y `y_train`.

Nuestro primer paso es un análisis exploratorio de datos profundo para entender la naturaleza de nuestras variables.

## 3.2. Análisis de las variables predictoras

#### 3.2.1. Numéricas

Ahora, veamos las variables numéricas en `X_train`. Usaremos `.describe()` para obtener estadísticas clave. Buscamos:

* **Escalas:** Diferencias drásticas en los rangos.
* **Valores extremos:** Diferencias grandes entre el 75% (`Q3`) y el máximo (`max`).
* **Asimetría:** Si la media es muy diferente de la mediana (`Q2`).

In [ ]:
include = ["number"]  # Data types to include
numerical_features = X_train.select_dtypes(include=include).columns.tolist()

In [ ]:
pd.set_option('display.max_rows', None)      # muestra todas las filas
pd.set_option('display.max_columns', None)   # muestra todas las columnas
pd.set_option('display.width', 2000)         # ancho total para evitar saltos
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.0f}'.format)
X_train[numerical_features].describe().T

La tabla `.describe()` ya nos da pistas claras.

Lo primero que observamos es el `count`: para todas las variables numéricas, el `count` coincide exactamente con el número total de instancias en nuestro conjunto de entrenamiento, lo que indica que no existen valores perdidos explícitos (`NaN`).

A continuación, vemos que, por ejemplo, `balance` tiene una media muy diferente de su mediana y un máximo lejísimos del 75%, lo que sugiere una fuerte **asimetría y valores extremos**.

Aunque estas estadísticas son increíblemente útiles para una detección rápida, no nos muestran la *forma* completa de la distribución. Un solo valor extremo puede sesgar la media, pero un histograma nos muestra *cuántos* valores extremos hay y cómo de "larga" es esa cola.

Para confirmar esto visualmente, podemos ayudarnos de **histogramas** (para ver la distribución y asimetría) y **diagramas de caja** (para identificar valores extremos).

In [ ]:
# Create plots
skipped = {}
const_cols = []
for column in numerical_features:
    try:
        # Extraer la columna
        x = X_train[column]

        # Comprobar que queda más de 1 valor
        if len(x) < 2:
            skipped[column] = "menos de 2 valores"
            continue

        # Evitar columnas constantes (boxplot rompe si no varían)
        if x.nunique() < 2:
            skipped[column] = "columna constante"
            continue

        # Graficar
        plt.figure(figsize=(12,5))

        # Histograma
        plt.subplot(1, 2, 1)
        sns.histplot(x, kde=True)
        plt.title("Histograma")

        # Boxplot
        plt.subplot(1, 2, 2)
        sns.boxplot(x=x)
        plt.title("Diagrama de caja")

        plt.suptitle(f'Análisis de la variable \"{column}\"')
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Error en la columna '{column}': {e}")

print("Columnas omitidas y razones:")
for col, reason in skipped.items():
    if reason == "columna constante":
        const_cols.append(col)
    print(f"- {col}: {reason}")

In [ ]:
X_src140 = X_train[X_train["src"] == 140]
skipped = {}
for column in numerical_features:
    try:
        # Extraer la columna
        x = X_src140[column]

        # Comprobar que queda más de 1 valor
        if len(x) < 2:
            skipped[column] = "menos de 2 valores"
            continue

        # Evitar columnas constantes (boxplot rompe si no varían)
        if x.nunique() < 2:
            skipped[column] = "columna constante"
            continue

        # Graficar
        plt.figure(figsize=(12,5))

        # Histograma
        plt.subplot(1, 2, 1)
        sns.histplot(x, kde=True)
        plt.title("Histograma")

        # Boxplot
        plt.subplot(1, 2, 2)
        sns.boxplot(x=x)
        plt.title("Diagrama de caja")

        plt.suptitle(f'Análisis de la variable \"{column}\"')
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"❌ Error en la columna '{column}': {e}")

print("Columnas omitidas y razones:")
for col, reason in skipped.items():
    print(f"- {col}: {reason}")

In [ ]:
X_src15 = X_train[X_train["src"] == 15]
skipped = {}
for column in numerical_features:
    try:
        # Extraer la columna
        x = X_src15[column]

        # Comprobar que queda más de 1 valor
        if len(x) < 2:
            skipped[column] = "menos de 2 valores"
            continue

        # Evitar columnas constantes (boxplot rompe si no varían)
        if x.nunique() < 2:
            skipped[column] = "columna constante"
            continue

        # Graficar
        plt.figure(figsize=(12,5))

        # Histograma
        plt.subplot(1, 2, 1)
        sns.histplot(x, kde=True)
        plt.title("Histograma")

        # Boxplot
        plt.subplot(1, 2, 2)
        sns.boxplot(x=x)
        plt.title("Diagrama de caja")

        plt.suptitle(f'Análisis de la variable \"{column}\"')
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"❌ Error en la columna '{column}': {e}")

print("Columnas omitidas y razones:")
for col, reason in skipped.items():
    print(f"- {col}: {reason}")

### 3.3. Análisis y manejo de valores faltantes

Como vimos antes, este conjunto de datos no tiene valores nulos explícitos (del tipo `NaN`). En su lugar, los valores faltantes están **"ocultos" bajo `"unknown"`**.

Ahora podemos usar `missingno` para visualizar la magnitud y la estructura de estas ausencias. En particular, usaremos **`msno.matrix()`**, que muestra una matriz de todo el conjunto de datos. Las barras negras representan datos presentes, y las **líneas blancas representan datos faltantes**. Nos permite ver patrones: ¿tienden a faltar valores en las mismas filas?

In [ ]:
plt.figure(figsize=(15,5))  # aumentar tamaño horizontal
msno.matrix(X_train)
plt.xticks(rotation=45)      # rotar nombres de columnas
plt.show()

In [ ]:
null_pct = X_train.isnull().mean() * 100
null_pct.plot(kind="bar", figsize=(15,5))
plt.ylabel("% de valores nulos")
plt.xticks(rotation=45)
plt.show()

In [ ]:
null_pct = X_train.isnull().mean() * 100  # porcentaje de nulos
null_summary = null_pct.sort_values(ascending=False).reset_index()
null_summary.columns = ["Variable", "% Nulos"]

print("Resumen de valores nulos por columna:")
print(null_summary.to_string(index=False))

Este gráfico de `missingno` es increíblemente claro y nos permite definir la estrategia final:

* **`poutcome`:** El gráfico de matriz muestra esta columna como casi completamente **blanca** (vacía).
    * **Estrategia:** **Eliminar la columna**. Es irrecuperable y no aporta información.

* **`contact`:** Vemos una cantidad significativa de líneas blancas. Crucialmente, la matriz **no** muestra una correlación fuerte con otros faltantes (es decir, las líneas blancas en `contact` no siempre coinciden con las de `education` o `job`).
    * **Estrategia:** La falta de este dato es probablemente informativa. No la imputaremos con la moda. La estrategia será **tratar el nulo como una categoría separada**.

* **`job` y `education`:** La matriz apenas muestra líneas blancas.
    * **Estrategia:** La cantidad es muy baja. Como en `contact`, **trataremos el nulo como una categoría separada**.

#### 3.2.2. Categóricas

Ahora, analizaremos las variables categóricas (`object`). Para estas, no nos interesa la media o la mediana, sino:

* **Cardinalidad:** Cuántos valores únicos (`unique`) tiene cada una. Una cardinalidad alta (muchas categorías) puede ser problemática.
* **Frecuencia:** Cuál es el valor más frecuente (`top`) y cuántas veces aparece (`freq`).
* **Valores faltantes ocultos:** Si el valor más frecuente (`top`) o la lista de valores únicos revela la presencia de `"unknown"` u otros marcadores.

In [ ]:
X_train['msg_class'] = X_train['msg_class'].astype('category')
X_train['msg_type'] = X_train['msg_type'].astype('category')
X_train['trace_options.trace_type'] = X_train['trace_options.trace_type'].astype('category')
X_train['configured'] = X_train['configured'].astype('category')
X_train['device_type'] = X_train['device_type'].astype('category')
X_train['error'] = X_train['error'].astype('category')
X_train['configuration'] = X_train['configuration'].astype('category')
X_train['inputState'] = X_train['inputState'].astype('category')
X_train['last_hw_test_result'] = X_train['last_hw_test_result'].astype('category')
X_train['last_sw_test_result'] = X_train['last_sw_test_result'].astype('category')
X_train['sensors'] = X_train['sensors'].astype('category')
X_train['pictogram'] = X_train['pictogram'].astype('category')
X_train['cbmac_packets_expired_pending'] = X_train['cbmac_packets_expired_pending'].astype('category')
X_train['Dropped_unack_bcs_packet'] = X_train['Dropped_unack_bcs_packet'].astype('category')
X_train['cbmac_broadcast_unack_pending'] = X_train['cbmac_broadcast_unack_pending'].astype('category')

include = ["object", "category"]  # Data types to include
categorical_features = X_train.select_dtypes(include=include).columns.tolist()
X_train[categorical_features].describe().T

### 3.4. Detección de duplicados e inconsistencias

En esta etapa se identifican registros duplicados y se verifican inconsistencias básicas, como formatos incorrectos o valores fuera de los rangos esperados, con el fin de garantizar la calidad y coherencia del conjunto de datos. Dado que ya se realizó una revisión de los rangos durante la exploración inicial de las variables, no es necesario repetir este análisis en esta fase.

In [ ]:
hashable_cols = [col for col in data.columns if data[col].apply(lambda x: isinstance(x, (int, float, str, type(None)))).all()]
duplicates = data[hashable_cols].duplicated()
num_duplicates = duplicates.sum()
print(f"Número de columnas hasheables {len(hashable_cols)}\nNúmero de filas duplicadas: {num_duplicates}")

Tal como se observa, no existen registros completamente duplicados en el conjunto de datos.

No obstante, también es posible identificar duplicados parciales, es decir, registros que comparten el mismo identificador pero presentan diferencias en otros campos.

In [ ]:
# Detect duplicate indices
keep = False  # Mark all duplicates as "True"
mask = data.index.duplicated(keep=keep)
duplicated_indices = data.index[mask]

# Filter records with duplicate indices
duplicated_records = data.loc[duplicated_indices]

# Verify if, for each duplicated index, the records are different
# Group by index and check if there are multiple unique rows
level = 0  # Group by index level
function = lambda group: group.drop_duplicates().shape[0] > 1
grouped = duplicated_records.groupby(level=level).filter(function)

# Display the found records
print("Duplicados parciales (mismo índice, datos diferentes):")
grouped

## 4. Modificación dataset



Eliminamos los atributos que tienen como nulos más del 32% de los valores


In [ ]:
null_over_32pct = null_pct[null_pct > 32].index.tolist()
print(f"Atributos totales: {len(X_train.columns)}\nAtributos con nulos < 32%:", len(X_train.columns) - len(null_over_32pct))

In [ ]:
X_train = X_train.drop(columns=null_over_32pct)
dropped_cols = set(null_over_32pct.copy())

Eliminamos los atributos que son constantes

In [ ]:
const_cols = []
for col in X_train.columns:
    if X_train[col].nunique() == 1:
        const_cols.append(col)

for col in null_over_32pct:
    if col in const_cols:
        const_cols.remove(col)
X_train = X_train.drop(columns=const_cols)
dropped_cols.update(const_cols)

Eliminamos los atributos que son casi consistentes

In [ ]:
cols_to_delete = set()
for col in X_train.columns: # eliminar columnas casi constantes
    freq = X_train[col].value_counts(normalize=True, dropna=False).values
    if freq[0] >= 0.95:
        cols_to_delete.add(col)

# cols_a_eliminar = [
#     'msg_class',
#     'msg_type',
#     'trace_options.trace_type',
#     'configured',
#     'device_type',
#     'configuration',
#     'inputState',
#     'last_hw_test_result',
#     'last_sw_test_result',
#     'sensors',
#     'pictogram',
#     'cbmac_packets_expired_pending',
#     'Dropped_unack_bcs_packet',
#     'cbmac_broadcast_unack_pending'
# ]
# cols_to_delete.update(cols_a_eliminar)


X_train = X_train.drop(columns=cols_to_delete)

Tratamos los valores nulos que aún existen en nuestro dataset

In [ ]:
for _, row in null_summary.iterrows():
    if row['Variable'] in dropped_cols:
        continue
    if row['% Nulos'] < 32:
        X_train[row['Variable']] = X_train[row['Variable']].fillna(X_train[row['Variable']].mode()[0])

Tratamos los valores extremos

In [ ]:
factor = 4 # Es el valor tradicional recomendado por Tukey. Un valor más alto es más conservador y elimina menos outliers
num_atr = X_train.select_dtypes(include=['number']).columns.tolist()

for col in num_atr:
    q1 = X_train[col].quantile(0.25)
    q3 = X_train[col].quantile(0.75)
    iqr = q3 - q1
    
    mask = (X_train[col] >= q1 - factor*iqr) & (X_train[col] <= q3 + factor*iqr) # Devuelve una mascara de booleanos que indica si el valor es o no un outlier
    X_train = X_train[mask] # Al aplicar la mascara se eliminan las columnas con false

In [ ]:
# Describimos el dataset modificado
print("Total number of atributes", len(X_train.columns))
print("Total number of records", len(X_train))
pd.set_option('display.max_rows', None)      # muestra todas las filas
pd.set_option('display.max_columns', None)   # muestra todas las columnas
pd.set_option('display.width', 2000)         # ancho total para evitar saltos
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.0f}'.format)
X_train.describe().T